In [8]:
# ── 셀 1: GPU 확인 ──────────────────────────────────────────
import torch
print('CUDA 사용 가능:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

CUDA 사용 가능: True
GPU: Tesla T4


In [9]:
# ── 셀 2: 패키지 설치 ────────────────────────────────────────
!pip install -q sentence-transformers FlagEmbedding

In [10]:
# ── 셀 3: chunks.json 업로드 ─────────────────────────────────
import os
if os.path.exists('chunks.json'):
    os.remove('chunks.json')
    print('기존 파일 삭제 완료')

from google.colab import files
uploaded = files.upload()  # chunks.json 선택

기존 파일 삭제 완료


Saving chunks.json to chunks.json


In [11]:
import json
with open('chunks.json', encoding='utf-8') as f:
    chunks = json.load(f)
print(len(chunks))
print(chunks[0])

5726
{'chunk_id': 'LCA_1', 'law_name': '지방계약법', 'article_id': '제1조', 'article_number': '제1조', 'text': '제1조(목적) 이 법은 지방자치단체를 당사자로 하는 계약에 관한 기본적인 사항을 정함으로써 계약업무를 원활하게 수행할 수 있도록 함을 목적으로 한다. [전문개정 2009. 2. 6.]', 'is_parent': False, 'parent_id': None, 'is_ref_article': False, 'is_upper_law': False, 'hierarchy': {'조': '제1조'}}


In [12]:
# ── 셀 4: 청크 로드 ──────────────────────────────────────────
# parent 청크(is_parent=True)는 검색 대상이 아니라
# Hierarchical RAG에서 child hit 후 fetch용이므로 임베딩에서 제외.
import json

with open('chunks.json', encoding='utf-8') as f:
    all_chunks = json.load(f)

# chunk_id 기준 중복 제거
seen = {}
for c in all_chunks:
    seen[c['chunk_id']] = c
all_chunks = list(seen.values())

# parent 제외 — 실제 검색 대상 child 청크만
search_chunks = [c for c in all_chunks if not c.get('is_parent', False)]

texts     = [c['text']     for c in search_chunks]
chunk_ids = [c['chunk_id'] for c in search_chunks]

print(f'전체 청크:         {len(all_chunks)}')
print(f'parent 청크:       {sum(1 for c in all_chunks if c.get("is_parent"))}')
print(f'임베딩 대상(child): {len(search_chunks)}')

전체 청크:         5726
parent 청크:       710
임베딩 대상(child): 5016


In [13]:
# ── 셀 5: 평가셋 생성 ────────────────────────────────────────
# parent 청크 제외 — child 청크만 평가 대상
import random
import re

eval_data = []
qid = 1

for chunk in search_chunks:
    text = chunk['text']
    if len(text) < 80:
        continue

    templates = [
        '다음 내용은 어떤 규정을 설명하나요?',
        '이 조항은 무엇에 관한 내용인가요?',
        '다음 규정과 관련된 조문을 찾아주세요.',
        '계약 실무에서 아래 내용은 어떤 법적 근거에 해당하나요?'
    ]

    query = random.choice(templates) + '\n\n' + text[:120]

    eval_data.append({
        'query_id':      f'eval_{qid:03d}',
        'query':         query,
        'relevant_docs': [chunk['chunk_id']]
    })
    qid += 1

random.shuffle(eval_data)
eval_data = eval_data[:200]

with open('eval_dataset_v2.json', 'w', encoding='utf-8') as f:
    json.dump(eval_data, f, ensure_ascii=False, indent=2)

print(f'평가셋 생성: {len(eval_data)}개')

평가셋 생성: 200개


In [14]:
# ── 셀 6: 모델 평가 함수 ─────────────────────────────────────
import gc
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

def evaluate_model(model_name, batch_size=8):
    print(f'\n===== {model_name} =====')

    model = SentenceTransformer(model_name, device='cuda')

    doc_vectors = model.encode(
        texts,
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True
    )

    recall1 = recall5 = recall10 = mrr = 0

    for item in eval_data:
        gt_docs = set(item['relevant_docs'])
        qvec = model.encode(
            item['query'],
            normalize_embeddings=True,
            convert_to_numpy=True
        )
        scores     = cosine_similarity([qvec], doc_vectors)[0]
        ranked_ids = [chunk_ids[i] for i in np.argsort(scores)[::-1]]

        if any(d in gt_docs for d in ranked_ids[:1]):  recall1  += 1
        if any(d in gt_docs for d in ranked_ids[:5]):  recall5  += 1
        if any(d in gt_docs for d in ranked_ids[:10]): recall10 += 1
        for rank, d in enumerate(ranked_ids, 1):
            if d in gt_docs:
                mrr += 1 / rank
                break

    n = len(eval_data)
    del doc_vectors, model
    gc.collect()
    torch.cuda.empty_cache()

    return {
        'Model':     model_name,
        'Recall@1':  round(recall1  / n, 4),
        'Recall@5':  round(recall5  / n, 4),
        'Recall@10': round(recall10 / n, 4),
        'MRR':       round(mrr      / n, 4),
    }

In [15]:
# ── 셀 7: BGE-M3 평가 ────────────────────────────────────────
import pandas as pd
result = evaluate_model('BAAI/bge-m3', batch_size=8)
display(pd.DataFrame([result]))


===== BAAI/bge-m3 =====


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/627 [00:00<?, ?it/s]

,Model,Recall@1,Recall@5,Recall@10,MRR
0,BAAI/bge-m3,0.835,0.945,0.99,0.8802


In [16]:
# ── 셀 8: KURE-v1 평가 ───────────────────────────────────────
result = evaluate_model('nlpai-lab/KURE-v1', batch_size=8)
display(pd.DataFrame([result]))


===== nlpai-lab/KURE-v1 =====


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/16.9k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Batches:   0%|          | 0/627 [00:00<?, ?it/s]

,Model,Recall@1,Recall@5,Recall@10,MRR
0,nlpai-lab/KURE-v1,0.845,0.965,1.0,0.8932


In [17]:
# ── 셀 9: Qwen3 평가 ─────────────────────────────────────────
result = evaluate_model('Qwen/Qwen3-Embedding-0.6B', batch_size=1)
display(pd.DataFrame([result]))


===== Qwen/Qwen3-Embedding-0.6B =====


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Batches:   0%|          | 0/5016 [00:00<?, ?it/s]

,Model,Recall@1,Recall@5,Recall@10,MRR
0,Qwen/Qwen3-Embedding-0.6B,0.79,0.96,0.995,0.862


In [18]:
# ── 셀 10: BGE-M3 dense + sparse 임베딩 및 저장 ──────────────
from FlagEmbedding import BGEM3FlagModel
import numpy as np
import json

print(f'임베딩 대상: {len(search_chunks)}개 (parent 제외)')

model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True)

output = model.encode(
    texts,
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=False,
    batch_size=8,
)

# dense 저장
np.savez(
    'vectors.npz',
    vectors=output['dense_vecs'],
    chunk_ids=np.array(chunk_ids)
)

# sparse 저장 — float16 → float 변환 후 직렬화
sparse_serializable = [
    {k: float(v) for k, v in sw.items()}
    for sw in output['lexical_weights']
]
with open('sparse_weights.json', 'w', encoding='utf-8') as f:
    json.dump(sparse_serializable, f)

print(f'✅ 저장 완료!')
print(f'   dense shape : {output["dense_vecs"].shape}')
print(f'   sparse 청크 수: {len(output["lexical_weights"])}')

from google.colab import files
files.download('vectors.npz')
files.download('sparse_weights.json')

임베딩 대상: 5016개 (parent 제외)


Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Inference Embeddings: 100%|██████████| 627/627 [00:17<00:00, 35.35it/s]


✅ 저장 완료!
   dense shape : (5016, 1024)
   sparse 청크 수: 5016


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>